In [ ]:
import pandas as pd
from pathlib import Path
import altair as alt

data = pd.read_csv("cleaned_listings.csv")

data.head()

print(data.isnull().sum())
data.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
price                   0
number_of_reviews       0
latitude                0
longitude               0
room_type               0
minimum_nights          0
availability_365        0
license              1127
has_license             0
dtype: int64


,price,number_of_reviews,latitude,longitude,room_type,minimum_nights,availability_365,license,has_license
0,125.00,26,42.36413,-71.02991,Entire home/apt,29,262,NaN,0
1,142.00,137,42.32844,-71.09581,Entire home/apt,3,103,STR-490093,1
2,171.00,139,42.32802,-71.09387,Entire home/apt,3,93,STR-491702,1
3,140.00,29,42.35867,-71.06307,Entire home/apt,91,210,NaN,0
4,152.00,9,42.35173,-71.08685,Entire home/apt,91,217,NaN,0


In [ ]:
import geopandas as gpd

filepath = "neighbourhoods.geojson"
neigh = gpd.read_file(filepath)
neigh_small = neigh[["neighbourhood", "geometry"]]

gdf_list = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data.longitude, data.latitude),
    crs="EPSG:4326"
)

joined = gpd.sjoin(gdf_list, neigh_small, how="left", predicate="within")

joined.head()

,price,number_of_reviews,latitude,longitude,room_type,minimum_nights,availability_365,license,has_license,geometry,index_right,neighbourhood
0,125.00,26,42.36413,-71.02991,Entire home/apt,29,262,NaN,0,POINT (-71.02991 42.36413),11,East Boston
1,142.00,137,42.32844,-71.09581,Entire home/apt,3,103,STR-490093,1,POINT (-71.09581 42.32844),8,Roxbury
2,171.00,139,42.32802,-71.09387,Entire home/apt,3,93,STR-491702,1,POINT (-71.09387 42.32802),8,Roxbury
3,140.00,29,42.35867,-71.06307,Entire home/apt,91,210,NaN,0,POINT (-71.06307 42.35867),14,Beacon Hill
4,152.00,9,42.35173,-71.08685,Entire home/apt,91,217,NaN,0,POINT (-71.08685 42.35173),10,Back Bay


In [ ]:
input_dropdown = alt.binding_select(options = [None, 'Roslindale', 'Jamaica Plain', 'Mission Hill',
       'Longwood Medical Area', 'Bay Village', 'Leather District',
       'Chinatown', 'North End', 'Roxbury', 'South End', 'Back Bay',
       'East Boston', 'Charlestown', 'West End', 'Beacon Hill',
       'Downtown', 'Fenway', 'Brighton', 'West Roxbury', 'Hyde Park',
       'Mattapan', 'Dorchester', 'South Boston Waterfront',
       'South Boston', 'Allston', 'Harbor Islands'],
                                    labels = ['All','Roslindale', 'Jamaica Plain', 'Mission Hill',
       'Longwood Medical Area', 'Bay Village', 'Leather District',
       'Chinatown', 'North End', 'Roxbury', 'South End', 'Back Bay',
       'East Boston', 'Charlestown', 'West End', 'Beacon Hill',
       'Downtown', 'Fenway', 'Brighton', 'West Roxbury', 'Hyde Park',
       'Mattapan', 'Dorchester', 'South Boston Waterfront',
       'South Boston', 'Allston', 'Harbor Islands'],
                                     name = "Neighbor: " )

selection = alt.selection_point(fields = ['neighbourhood'], bind = input_dropdown)

alt.Chart(joined).mark_bar().transform_filter(selection).encode(
    x = alt.X('has_license:N'),
    y = alt.Y('count(has_license):Q'),
).add_params(
    selection
)

alt.Chart(...)

In [ ]:
input_dropdown = alt.binding_select(
    options=[None, 'Roslindale', 'Jamaica Plain', 'Mission Hill',
             'Longwood Medical Area', 'Bay Village', 'Leather District',
             'Chinatown', 'North End', 'Roxbury', 'South End', 'Back Bay',
             'East Boston', 'Charlestown', 'West End', 'Beacon Hill',
             'Downtown', 'Fenway', 'Brighton', 'West Roxbury', 'Hyde Park',
             'Mattapan', 'Dorchester', 'South Boston Waterfront',
             'South Boston', 'Allston', 'Harbor Islands'],
    labels=['All','Roslindale', 'Jamaica Plain', 'Mission Hill',
            'Longwood Medical Area', 'Bay Village', 'Leather District',
            'Chinatown', 'North End', 'Roxbury', 'South End', 'Back Bay',
            'East Boston', 'Charlestown', 'West End', 'Beacon Hill',
            'Downtown', 'Fenway', 'Brighton', 'West Roxbury', 'Hyde Park',
            'Mattapan', 'Dorchester', 'South Boston Waterfront',
            'South Boston', 'Allston', 'Harbor Islands'],
    name="Neighbor: "
)

selection = alt.selection_point(fields=['neighbourhood'], bind=input_dropdown)

base = (
    alt.Chart(joined)
    .add_params(selection)
    .transform_filter(selection)
)

joined['price'] = pd.to_numeric(joined['price'], errors='coerce')

count_bar = base.mark_bar().encode(
    x=alt.X('has_license:N', title='Has License'),
    y=alt.Y('count():Q', title='Count of Listings'),
    color='has_license:N',
    tooltip=['has_license:N', 'count():Q']
).properties(title='Licensed vs Unlicensed (Count)')


price_bar = base.mark_bar().encode(
    x=alt.X('has_license:N', title='Has License'),
    y=alt.Y('mean(price):Q', title='Average Price'),
    color='has_license:N',
    tooltip=['has_license:N', 'mean(price):Q']
).properties(title='Licensed vs Unlicensed (Average Price)')

final_chart = count_bar | price_bar

c_json = final_chart.to_json()

with open('website/t5-barplot1_spec.json', 'w') as f:
    f.write(c_json)

alt.HConcatChart(...)